# triangle-splatting :: Custom data :: WhitePass_Train

-----
- Conda env : [waikiki_statue](README.md#setup-a-conda-environment)
-----

### Check system

In [1]:
!nvidia-smi

Sat Oct  4 09:48:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 36%   43C    P5             33W /  250W |     555MiB /  11264MiB |     40%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [2]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [3]:
import gdown
VIDEO_NAME = "WhitePass_Train"

FPS = 10
RES = 4
id = "1LdXaVkme3egIwiMMevApDnwZcMUv0VvK"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"

gdown.download(id=id, output = vid_path)

Downloading...
From: https://drive.google.com/uc?id=1LdXaVkme3egIwiMMevApDnwZcMUv0VvK
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/triangle_splatting/temp_data/WhitePass_Train.mov
100%|██████████| 60.7M/60.7M [00:05<00:00, 11.7MB/s]


'./temp_data/WhitePass_Train.mov'

### Extract images from the video

In [4]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)


!ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Colmap :: Feature Extraction

In [5]:
# Colmap Feature Extraction
!colmap feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE


Feature extraction

Processed file [1/554]
  Name:            frame_0001.jpg
  Dimensions:      1080 x 1920
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        4672
Processed file [2/554]
  Name:            frame_0002.jpg
  Dimensions:      1080 x 1920
  Camera:          #2 - PINHOLE
  Focal Length:    2304.00px
  Features:        4497
Processed file [3/554]
  Name:            frame_0003.jpg
  Dimensions:      1080 x 1920
  Camera:          #3 - PINHOLE
  Focal Length:    2304.00px
  Features:        4558
Processed file [4/554]
  Name:            frame_0004.jpg
  Dimensions:      1080 x 1920
  Camera:          #4 - PINHOLE
  Focal Length:    2304.00px
  Features:        4592
Processed file [5/554]
  Name:            frame_0005.jpg
  Dimensions:      1080 x 1920
  Camera:          #5 - PINHOLE
  Focal Length:    2304.00px
  Features:        4633
Processed file [6/554]
  Name:            frame_0006.jpg
  Dimensions:      1080 x 1920
  Camera:          #6 - PI

### Colmap :: Feature Matching

In [6]:
# Feature Matching
!colmap sequential_matcher \
    --database_path $DATABASE_PATH


Sequential feature matching

Matching image [1/554] in 0.302s
Matching image [2/554] in 0.149s
Matching image [3/554] in 0.174s
Matching image [4/554] in 0.172s
Matching image [5/554] in 0.279s
Matching image [6/554] in 0.166s
Matching image [7/554] in 0.179s
Matching image [8/554] in 0.392s
Matching image [9/554] in 0.164s
Matching image [10/554] in 0.168s
Matching image [11/554] in 0.274s
Matching image [12/554] in 0.167s
Matching image [13/554] in 0.325s
Matching image [14/554] in 0.153s
Matching image [15/554] in 0.208s
Matching image [16/554] in 0.151s
Matching image [17/554] in 0.208s
Matching image [18/554] in 0.153s
Matching image [19/554] in 0.162s
Matching image [20/554] in 0.201s
Matching image [21/554] in 0.186s
Matching image [22/554] in 0.172s
Matching image [23/554] in 0.250s
Matching image [24/554] in 0.197s
Matching image [25/554] in 0.187s
Matching image [26/554] in 0.261s
Matching image [27/554] in 0.190s
Matching image [28/554] in 0.216s
Matching image [29/554] in 

### Colmap :: Sparse Reconstruction (Mapper)

In [7]:
# Sparse Reconstruction (Mapper)
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

!colmap mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH \
    --output_path $SPARSE_DIR


Loading database

Loading cameras... 554 in 0.001s
Loading matches... 5806 in 0.081s
Loading images... 554 in 0.069s (connected 554)
Building correspondence graph... in 0.405s (ignored 0)

Elapsed time: 0.009 [minutes]


Finding good initial image pair


Initializing with image pair #320 and #352


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  6.632047e+01    0.00e+00    7.80e+02   0.00e+00   0.00e+00  1.00e+04        0    1.79e-04    8.13e-04
   1  6.413838e+01    2.18e+00    2.69e+02   1.03e+01   9.84e-01  3.00e+04        1    3.31e-04    1.16e-03
   2  6.344805e+01    6.90e-01    1.22e+03   1.64e+01   9.88e-01  9.00e+04        1    2.54e-04    1.42e-03
   3  6.404882e+01   -6.01e-01    1.22e+03   3.97e+01  -1.53e+00  4.50e+04        1    2.40e-04    1.66e-03
   4  6.325341e+01    1.95e-01    3.81e+03   2.04e+01   7.46e-01  5.11e+04        1    2.78e-04    1.95e-03
   5  6.311999e+01    1.3

### Triangle-Splatting :: Training (Indoor mode)

In [8]:
DATASET_DIR_PATH
OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}"
print(DATASET_DIR_PATH)
print(OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTPUR_DIR_PATH -r $RES --eval

./temp_data/WhitePass_Train
./temp_result/WhitePass_Train
Optimizing ./temp_result/WhitePass_Train
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output folder: 

### Triangle-Splatting :: Rendering (Indoor mode)

In [9]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/WhitePass_Train/cfg_args
Config file found: ./temp_result/WhitePass_Train/cfg_args
Rendering ./temp_result/WhitePass_Train
Loading trained model at iteration 7000 [04/10 10:33:57]
Reading camera 1/550----- PINHOLE [04/10 10:33:58]
Reading camera 2/550----- PINHOLE [04/10 10:33:58]
Reading camera 3/550----- PINHOLE [04/10 10:33:58]
Reading camera 4/550----- PINHOLE [04/10 10:33:58]
Reading camera 5/550----- PINHOLE [04/10 10:33:58]
Reading camera 6/550----- PINHOLE [04/10 10:33:58]
Reading camera 7/550----- PINHOLE [04/10 10:33:58]
Reading camera 8/550----- PINHOLE [04/10 10:33:58]
Reading camera 9/550----- PINHOLE [04/10 10:33:58]
Reading camera 10/550----- PINHOLE [04/10 10:33:58]
Reading camera 11/550----- PINHOLE [04/10 10:33:58]
Reading camera 12/550----- PINHOLE [04/10 10:33:58]
Reading camera 13/550----- PINHOLE [04/10 10:33:58]
Reading camera 14/550----- PINHOLE [04/10 10:33:58]
Reading camera 15/550----- PINHOLE [04/10 10:33:58]
Reading 

### Triangle-Splatting :: Create a video (Indoor mode)

In [10]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/WhitePass_Train/cfg_args
Config file found: ./temp_result/WhitePass_Train/cfg_args
Creating video for ./temp_result/WhitePass_Train
Loading trained model at iteration 7000
Reading camera 1/550----- PINHOLE
Reading camera 2/550----- PINHOLE
Reading camera 3/550----- PINHOLE
Reading camera 4/550----- PINHOLE
Reading camera 5/550----- PINHOLE
Reading camera 6/550----- PINHOLE
Reading camera 7/550----- PINHOLE
Reading camera 8/550----- PINHOLE
Reading camera 9/550----- PINHOLE
Reading camera 10/550----- PINHOLE
Reading camera 11/550----- PINHOLE
Reading camera 12/550----- PINHOLE
Reading camera 13/550----- PINHOLE
Reading camera 14/550----- PINHOLE
Reading camera 15/550----- PINHOLE
Reading camera 16/550----- PINHOLE
Reading camera 17/550----- PINHOLE
Reading camera 18/550----- PINHOLE
Reading camera 19/550----- PINHOLE
Reading camera 20/550----- PINHOLE
Reading camera 21/550----- PINHOLE
Reading camera 22/550----- PINHOLE
Reading camera 23/550-----

### Triangle-Splatting :: Training (Outdoor mode)

In [11]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval  --outdoor 

./temp_data/WhitePass_Train
./temp_result/WhitePass_Train_outdoor
Optimizing ./temp_result/WhitePass_Train_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth

### Triangle-Splatting :: Rendering (Outdoor mode)

In [12]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/WhitePass_Train_outdoor/cfg_args
Config file found: ./temp_result/WhitePass_Train_outdoor/cfg_args
Rendering ./temp_result/WhitePass_Train_outdoor
Loading trained model at iteration 7000 [04/10 10:49:31]
Reading camera 1/550----- PINHOLE [04/10 10:49:31]
Reading camera 2/550----- PINHOLE [04/10 10:49:31]
Reading camera 3/550----- PINHOLE [04/10 10:49:31]
Reading camera 4/550----- PINHOLE [04/10 10:49:31]
Reading camera 5/550----- PINHOLE [04/10 10:49:31]
Reading camera 6/550----- PINHOLE [04/10 10:49:31]
Reading camera 7/550----- PINHOLE [04/10 10:49:31]
Reading camera 8/550----- PINHOLE [04/10 10:49:31]
Reading camera 9/550----- PINHOLE [04/10 10:49:31]
Reading camera 10/550----- PINHOLE [04/10 10:49:31]
Reading camera 11/550----- PINHOLE [04/10 10:49:31]
Reading camera 12/550----- PINHOLE [04/10 10:49:31]
Reading camera 13/550----- PINHOLE [04/10 10:49:31]
Reading camera 14/550----- PINHOLE [04/10 10:49:31]
Reading camera 15/550----- PINHOLE [

### Triangle-Splatting :: Create a video (Outdoor mode)

In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/WhitePass_Train_outdoor/cfg_args
Config file found: ./temp_result/WhitePass_Train_outdoor/cfg_args
Creating video for ./temp_result/WhitePass_Train_outdoor
Loading trained model at iteration 7000
Reading camera 1/550----- PINHOLE
Reading camera 2/550----- PINHOLE
Reading camera 3/550----- PINHOLE
Reading camera 4/550----- PINHOLE
Reading camera 5/550----- PINHOLE
Reading camera 6/550----- PINHOLE
Reading camera 7/550----- PINHOLE
Reading camera 8/550----- PINHOLE
Reading camera 9/550----- PINHOLE
Reading camera 10/550----- PINHOLE
Reading camera 11/550----- PINHOLE
Reading camera 12/550----- PINHOLE
Reading camera 13/550----- PINHOLE
Reading camera 14/550----- PINHOLE
Reading camera 15/550----- PINHOLE
Reading camera 16/550----- PINHOLE
Reading camera 17/550----- PINHOLE
Reading camera 18/550----- PINHOLE
Reading camera 19/550----- PINHOLE
Reading camera 20/550----- PINHOLE
Reading camera 21/550----- PINHOLE
Reading camera 22/550----- PINHOLE
Re